<a href="https://colab.research.google.com/github/gav-ip/ML-zero/blob/main/Copy_of_transformer_addition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [59]:
import jax
from jax import random
import jax.numpy as jnp
import flax.nnx as nnx
import numpy as np
import random as py_random
import optax

In [60]:
USE_GPU = True
device = jax.devices("cpu")[0] if not USE_GPU else jax.devices()[0]
n_examples = 100000
block_size = 11
batch_size = 256
eval_iters = 200
max_iters = 5000
eval_interval = 500
n_embed = 320
n_heads = 4
n_layer = 8
dropout = 0.2
learning_rate = 3e-4

print(device)

cuda:0


In [61]:
vocab = {0:'0', 1:'1', 2:'2', 3:'3', 4:'4', 5:'5', 6:'6', 7:'7', 8:'8', 9:'9', 10:'+', 11:'=', 12:' '}
vocab_size = len(vocab)

In [62]:
stoi = {vocab[i]:i for i, ch in enumerate(vocab)}
itos = {i:vocab[i] for i, ch in enumerate(vocab)}
encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join([itos[int(i)] for i in l])

print(encode("8+2 =10 "))
print(decode(encode("8+7 =15 ")))

[8, 10, 2, 12, 11, 1, 0, 12]
8+7 =15 


In [63]:
!wget https://raw.githubusercontent.com/gav-ip/ML-zero/main/transformer-adder/addition.txt

--2026-09-09 22:27:59--  https://raw.githubusercontent.com/gav-ip/ML-zero/main/transformer-adder/addition.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1300000 (1.2M) [text/plain]
Saving to: ‘addition.txt.2’

addition.txt.2      100%[===================>]   1.24M  --.-KB/s    in 0.006s  

2026-09-09 22:27:59 (206 MB/s) - ‘addition.txt.2’ saved [1300000/1300000]



In [64]:
with open("addition.txt", "r", encoding="utf-8") as f:
    text = [line.rstrip("\n") for line in f]

encoded_text = [encode(t) for t in text]
data = jnp.array(encoded_text, device=device)
data[:3]

Array([[ 6,  3,  6, 10,  1,  7,  4, 11,  0,  1,  8, 12],
       [ 6,  7,  3, 10,  2,  0,  9, 11,  2,  8,  8, 12],
       [12, 12,  3, 10, 12, 12,  5, 11,  8, 12, 12, 12]], dtype=int32)

In [65]:
print(data[:5])
print(decode(data[0]))

[[ 6  3  6 10  1  7  4 11  0  1  8 12]
 [ 6  7  3 10  2  0  9 11  2  8  8 12]
 [12 12  3 10 12 12  5 11  8 12 12 12]
 [ 1  4  4 10  7  1  9 11  3  6  8 12]
 [12  1  4 10 12  5  5 11  9  6 12 12]]
636+174=018 


In [66]:
n = int(0.9*len(data)) # first 90% will be train, rest val
train_data = data[:n]
val_data = data[n:]

In [67]:
def get_batch(key, split):
    data = train_data if split == 'train' else val_data

    ix = random.randint(key, (batch_size,), 0, len(data))
    seq = data[ix]

    x = seq[:, :-1]
    y = seq[:, 1:]
    y = y.at[:,:7].set(-1)

    return x, y

In [68]:
key = random.PRNGKey(0)

# call jax.lax.stop_gradient for no_grad
def estimate_loss(key):
    out = {}
    model.eval()
    for split in ['train', 'val']:
         losses = jnp.zeros(eval_iters)
         for k in range(eval_iters):
             key, subkey = random.split(key)
             X, Y = get_batch(subkey, split)
             _, loss = model(X, Y)
             losses = losses.at[k].set(loss)
         out[split] = losses.mean()
    model.train()
    return out

In [69]:
rngs = nnx.Rngs(0)

class MultiHeadAttention(nnx.Module):
    def __init__(self, *, rngs: nnx.Rngs):
        self.key = nnx.Linear(n_embed, n_embed, use_bias=False, rngs=rngs)
        self.query = nnx.Linear(n_embed, n_embed, use_bias=False, rngs=rngs)
        self.value = nnx.Linear(n_embed, n_embed, use_bias=False, rngs=rngs)
        self.proj = nnx.Linear(n_embed, n_embed, rngs=rngs)
        self.dropout = nnx.Dropout(dropout, rngs=rngs)

    def __call__(self, x):
        B, T, C = x.shape
        head_dim = C // n_heads
        k = self.key(x).reshape(B, T, n_heads, head_dim)
        q = self.query(x).reshape(B, T, n_heads, head_dim)
        v = self.value(x).reshape(B, T, n_heads, head_dim)

        out = jnn.dot_product_attention(q, k, v, mask=None, is_causal=True)
        out = out.reshape(B, T, C)

        return self.dropout(self.proj(out))

class FeedForward(nnx.Module):
    def __init__(self, x: n_embed, *, rngs: nnx.Rngs):
        self.net = nnx.Sequential(
            nnx.Linear(x, 4 * x, rngs=rngs),
            nnx.gelu,
            nnx.Linear(4 * x, x, rngs=rngs),
            nnx.Dropout(dropout, rngs=rngs),
        )

    def __call__(self, x):
        return self.net(x)

class Block(nnx.Module):
    def __init__(self, n_embd, n_head, *, rngs: nnx.Rngs):
        self.sa = MultiHeadAttention(rngs=rngs)
        self.ff = FeedForward(n_embd, rngs=rngs)
        self.ln1 = nnx.LayerNorm(n_embd, rngs=rngs)
        self.ln2 = nnx.LayerNorm(n_embd, rngs=rngs)

    def __call__(self, x):
        x = x + self.sa(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x

class AdditionModel(nnx.Module):
    def __init__(self, rngs: nnx.Rngs):
        super().__init__()
        self.token_embedding_table = nnx.Embed(vocab_size, n_embed, rngs=rngs)
        self.position_embedding_table = nnx.Embed(block_size, n_embed, rngs=rngs)
        self.blocks = nnx.Sequential(
            *[Block(n_embed, n_heads, rngs=rngs) for _ in range(n_layer)]
        )
        self.ln_f = nnx.LayerNorm(n_embed, rngs=rngs)
        self.lm_head = nnx.Linear(n_embed, vocab_size, rngs=rngs)

    def __call__(self, idx, targets=None):

        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(jnp.arange(T, device=device)) # (T,C)
        x = tok_emb + pos_emb # (B,T,C)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x) # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.reshape(B*T, C)

            targets = targets.reshape(B*T)
            valid = targets != -1
            safe_targets = jnp.clip(targets, 0)

            token_loss = optax.softmax_cross_entropy_with_integer_labels(logits, safe_targets)
            loss = (token_loss * valid).sum() / jnp.maximum(valid.sum(), 1)

        return logits, loss

    def genereate(self, idx, max_new_tokens):

        for _ in range(max_new_tokens):

            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]

            key, subkey = random.split(key)
            idx_next = jax.random.categorical(subkey, logits)

            idx = jax.concatenate((idx, idx_next), axis=1)

        return idx

In [70]:
model = AdditionModel(rngs=nnx.Rngs(0))

In [71]:
params = nnx.state(model, nnx.Param)
param_count = sum(x.size for x in jax.tree_util.tree_leaves(params))
print(f"Total parameters: {param_count}")

Total parameters: 9868493


In [ ]:
optimizer = nnx.Optimizer(model, optax.adamw(learning_rate=learning_rate), wrt=nnx.Param)

# training loop
key = random.PRNGKey(1)

for steps in range(max_iters):
    key, eval_key, batch_key = random.split(key, 3)

    if steps % eval_interval == 0:
        losses = estimate_loss(eval_key)
        print(f"step {steps}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    # get sample batch from training set
    xb, yb = get_batch(key,'train')

    def compute_train_loss(current_model, x, y):
        _, loss_value = current_model(x, y)
        return loss_value

    # Evaluate loss and compute gradients
    # The first return value is the scalar loss, the second are the gradients
    loss, grads = nnx.value_and_grad(compute_train_loss)(model, xb, yb)
    optimizer.update(model, grads)

In [ ]:
import pickle

checkpoint = {
    'model_state_dict': nnx.state_dict(),
    'stoi': stoi,
    'itos': itos,
    'config': {
        'vocab_size': vocab_size,
        'block_size': block_size,
        'n_embd': n_embed,
        'n_head': n_heads,
        'n_layer': n_layer,
        'dropout': dropout,
    }
}
with open('nano_gpt.pkl', 'wb') as f:
    pickle.dump(checkpoint, f)